# The same model, the same problem, 20 times

A reasoning model's score on a benchmark hides how _unstable_ a single attempt is.
Here we take **one** Game24 puzzle — _make 24 from the numbers 6, 6, 7, 12_ — and
look at **gpt-5-mini** solving it 20 separate times under identical settings.

Game24 is mechanically checkable: a valid answer must use each input number exactly
once and evaluate to 24, so we can grade every attempt. The reconstruction (parsing
the shipped trace logs + grading) lives in `instability.py`.


In [4]:
import attempts as I


# "deepseek-r1", (6,11,12,13)
# "gpt-5-mini", puzzle=(6, 6, 7, 12)

I.show("deepseek-r1", (6,11,12,13))

(13 - 11) * 6 + 12,×7
((6 * (13 - 11)) + 12),×2
((13 - 11) * 6) + 12,×1
(12 + 6 * (13 - 11)),×1
(6 * (13 - 11) + 12),×1
((13 - 11) * 6 + 12),×1
(24 * 1),invents an extra 1; invents an extra 24; never uses 6; never uses 11; never uses 12; never uses 13
(12 + 12),invents an extra 12; never uses 6; never uses 11; never uses 13
(6 * (13 - (11 - 12 / 6))),invents an extra 6
(12 * (13 - 11)) * (6 / 6),invents an extra 6
0** 4. **24 + 0,no parseable expression


**What to take away.** The headline number — _45% accuracy_ — is the least
interesting thing here. The shape of the runs is the point:

- **It's a coin toss per call.** 9 of 20 identical requests land a valid answer; nothing
  changed but the sampling seed.
- **One truth, many lies.** Every correct run rediscovers the _same single idea_
  (42 − 6 − 12). The 11 failures are 11 _different_ confident, well-formed,
  wrong expressions.
- **The failures aren't random noise — they cheat the same way.** Almost every wrong
  answer **fabricates a number it wasn't given**, slipping in `6/6 = 1` (or `7/7`,
  `6−6`) to absorb the leftover operand and force the total to 24.

This per-call instability is exactly what test-time scaling (sampling + verification
or self-consistency) is paying to average out.


Try other models / puzzles (each has its own worst case in the logs):

```python
I.show("deepseek-r1", (6, 11, 12, 13))   # a top reasoning model does it too
I.show("gpt-4.1-mini", (6, 7, 8, 9))     # an exact 10/10 coin flip
```

For the raw per-attempt table without the HTML, run `instability.py` directly:
`python instability.py`.
